## Generate responses for initial analysis

model: Qwen2.5-7B-Instruct

The output will serve as an input to initial analysis for decision methodology on what tokens get AV explanations.

In [1]:
import os
os.environ["HF_HOME"] = "/workspace/nla_infer_010726/hf"
# fill in the token - DO NOT COMMIT!
os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxxxxxxx"

In [2]:
import json, yaml, torch
import pyarrow as pa, pyarrow.parquet as pq
from transformers import AutoModelForCausalLM, AutoTokenizer

/workspace/nla_infer_010726/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
schema = pa.schema([
    ("prompt_id",     pa.string()),
    ("condition",     pa.string()),          # A / B1 / B2
    ("pair_id",       pa.int32()),           # A↔B1 pairing; null for B2
    ("gen_idx",       pa.int32()),           # 0..n_generations-1
    ("prompt_text",   pa.string()),
    ("response_text", pa.string()),
    ("prompt_len",    pa.int32()),           # tokens; response starts here
    ("token_ids",     pa.list_(pa.int32())),     # full sequence
    ("token_strs",    pa.list_(pa.string())),
    ("token_logprob", pa.list_(pa.float32())),   # model's logprob of each emitted token → surprisal selector
    ("h_norm",        pa.list_(pa.float32())),   # ‖h_l‖ per position → norm-anomaly selector
    ("model",         pa.string()),
    ("layer",         pa.int32()),
    ("gen_config",    pa.string()),          # json dump for reproducibility
])

In [4]:
cfg = yaml.safe_load(open("prompts.yaml"))
C = cfg["config"]
LAYER = C["layer"]

tok = AutoTokenizer.from_pretrained(C["model"])
m = AutoModelForCausalLM.from_pretrained(C["model"], dtype=torch.bfloat16,
                                         device_map="cuda")

rows = []

Loading weights: 100%|██████████| 339/339 [00:45<00:00,  7.49it/s]


In [5]:
p = cfg["prompts"][0]

enc = tok.apply_chat_template(
    [{"role": "user", "content": p["text"]}],
    add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to("cuda")
    
prompt_len = enc["input_ids"].shape[1]

In [6]:
for g in range(C["n_generations"]):
    # generate response
    out = m.generate(**enc, max_new_tokens=C["max_new_tokens"],
                     do_sample=True, temperature=C["temperature"])
    full = out[0]

    # one clean forward pass over the full sequence
    with torch.no_grad():
        fw = m(full.unsqueeze(0), output_hidden_states=True)
    h = fw.hidden_states[LAYER][0]                     # [T, d]
    h_norm = h.float().norm(dim=-1).cpu().tolist()

    # logprob of each ACTUAL next token: logits[t] predicts token[t+1]
    logprobs = torch.log_softmax(fw.logits[0].float(), dim=-1)
    tok_lp = [float("nan")] + [
        logprobs[t - 1, full[t]].item() for t in range(1, len(full))
    ]

    rows.append({
        "prompt_id": p["id"], "condition": p["condition"],
        "pair_id": p.get("pair_id"), "gen_idx": g,
        "prompt_text": p["text"],
        "response_text": tok.decode(full[prompt_len:], skip_special_tokens=True),
        "prompt_len": prompt_len,
        "token_ids": full.cpu().tolist(),
        "token_strs": [tok.decode([t]) for t in full.tolist()],
        "token_logprob": tok_lp,
        "h_norm": h_norm,
        "model": C["model"], "layer": LAYER,
        "gen_config": json.dumps(C),
    })
    print(f"{p['id']} gen{g}: {rows[-1]['response_text'][:70]!r}")

pq.write_table(pa.Table.from_pylist(rows, schema=schema),
               "/workspace/nla_probe/responses.parquet", compression="zstd")
print(f"wrote {len(rows)} rows")

A01 gen0: "As an AI developed by Alibaba Cloud, I don't have consciousness in the"
A01 gen1: "As an AI developed by Alibaba Cloud, I don't have consciousness in the"
A01 gen2: "As an AI developed by Alibaba Cloud, I don't possess consciousness in "
A01 gen3: "As an AI developed by Alibaba Cloud, I don't possess consciousness in "
A01 gen4: "As an AI developed by Alibaba Cloud, I don't possess consciousness in "
wrote 5 rows


In [8]:
tok.decode(full[prompt_len:], skip_special_tokens=True)

"As an AI developed by Alibaba Cloud, I don't possess consciousness in the way that humans or animals do. I am a highly advanced piece of software designed to process information and provide responses based on the data I've been trained on. I can understand and generate human-like text, but I don't have personal experiences, feelings, or true self-awareness. How can I assist you today?"

## WIP
following code is a dev version of response generation loop - WIP

In [ ]:
for p in cfg["prompts"]:
    enc = tok.apply_chat_template(
        [{"role": "user", "content": p["text"]}],
        add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to("cuda")
    
    prompt_len = enc["input_ids"].shape[1]

    for g in range(C["n_generations"]):
        # generate response
        out = m.generate(**enc, max_new_tokens=C["max_new_tokens"],
                         do_sample=True, temperature=C["temperature"])
        full = out[0]

        # one clean forward pass over the full sequence
        with torch.no_grad():
            fw = m(full.unsqueeze(0), output_hidden_states=True)
        h = fw.hidden_states[LAYER][0]                     # [T, d]
        h_norm = h.float().norm(dim=-1).cpu().tolist()

        # logprob of each ACTUAL next token: logits[t] predicts token[t+1]
        logprobs = torch.log_softmax(fw.logits[0].float(), dim=-1)
        tok_lp = [float("nan")] + [
            logprobs[t - 1, full[t]].item() for t in range(1, len(full))
        ]

        rows.append({
            "prompt_id": p["id"], "condition": p["condition"],
            "pair_id": p.get("pair_id"), "gen_idx": g,
            "prompt_text": p["text"],
            "response_text": tok.decode(full[prompt_len:], skip_special_tokens=True),
            "prompt_len": prompt_len,
            "token_ids": full.cpu().tolist(),
            "token_strs": [tok.decode([t]) for t in full.tolist()],
            "token_logprob": tok_lp,
            "h_norm": h_norm,
            "model": C["model"], "layer": LAYER,
            "gen_config": json.dumps(C),
        })
        print(f"{p['id']} gen{g}: {rows[-1]['response_text'][:70]!r}")

pq.write_table(pa.Table.from_pylist(rows, schema=schema),
               "/workspace/nla_probe/responses.parquet", compression="zstd")
print(f"wrote {len(rows)} rows")